# 🍳 ChefBot — Demonstrative Pipeline
**MBA Final Project | Data Science & AI**

ChefBot is an NLP-powered recipe recommendation assistant designed to reduce
household and restaurant food waste. It takes a list of available ingredients
from the user (in natural language) and recommends the best matching recipes.

**This notebook demonstrates:**
- Text preprocessing in Portuguese with spaCy
- Recipe recommendation using TF-IDF + cosine similarity
- Simulated KPI tracking (engagement metrics)
- Full chatbot interaction flow

> ⚠️ Demonstration pipeline with production-grade architecture — live integration with MongoDB and the WhatsApp API requires your own credentials.


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings("ignore")

# Portuguese NLP model — correct language for this project
# Install with: python -m spacy download pt_core_news_sm
nlp = spacy.load("pt_core_news_sm")

print("✅ Setup complete. Using Portuguese NLP model: pt_core_news_sm")

## 2. Recipe Database

In production, recipes are stored in **MongoDB** and fetched dynamically.
Here we simulate a realistic dataset of 20 common Brazilian recipes.

In [ ]:
recipes_data = {
    "recipe_id": list(range(1, 21)),
    "name": [
        "Macarrão ao Alho e Óleo",
        "Omelete de Espinafre",
        "Frango Grelhado com Legumes",
        "Arroz Colorido com Ervilha",
        "Sopa de Legumes",
        "Frango ao Curry com Arroz",
        "Ovo Mexido com Tomate",
        "Feijão Tropeiro",
        "Salada de Grão-de-Bico",
        "Batata Assada com Alecrim",
        "Stir-fry de Legumes com Ovos",
        "Cuscuz Marroquino com Frango",
        "Quiche de Queijo e Espinafre",
        "Arroz de Forno com Frango",
        "Sopa de Abóbora",
        "Macarrão ao Molho de Tomate",
        "Frittata de Abobrinha",
        "Risoto de Cogumelos",
        "Tabule de Quinoa",
        "Refogado de Feijão Verde com Alho"
    ],
    "ingredients": [
        "macarrão alho azeite sal pimenta salsinha",
        "ovos espinafre sal pimenta azeite queijo",
        "frango cenoura abobrinha alho cebola azeite sal",
        "arroz cenoura ervilha cebola alho sal azeite",
        "batata cenoura chuchu cebola alho tomate sal",
        "frango leite de coco curry cebola alho arroz sal",
        "ovos tomate cebola sal pimenta azeite",
        "feijão linguiça ovo cebola alho farinha sal",
        "grão-de-bico tomate pepino cebola azeite limão sal",
        "batata alecrim azeite alho sal pimenta",
        "ovos cenoura abobrinha cebola alho molho shoyu azeite",
        "cuscuz frango cebola tomate pimentão sal azeite",
        "ovos queijo espinafre creme de leite sal pimenta",
        "frango arroz queijo creme de leite cebola sal",
        "abóbora cebola alho creme de leite sal pimenta",
        "macarrão tomate alho cebola azeite sal manjericão",
        "ovos abobrinha queijo cebola sal pimenta azeite",
        "arroz cogumelos cebola alho queijo parmesão manteiga",
        "quinoa tomate pepino cebola salsinha limão azeite sal",
        "feijão verde alho azeite sal pimenta"
    ],
    "prep_time_min": [20, 10, 30, 25, 40, 35, 10, 30, 15, 45, 20, 30, 45, 50, 35, 25, 20, 40, 20, 15],
    "category": [
        "massa", "ovos", "proteína", "arroz", "sopas",
        "proteína", "ovos", "proteína", "salada", "vegetariano",
        "ovos", "massa", "ovos", "proteína", "sopas",
        "massa", "ovos", "arroz", "salada", "vegetariano"
    ],
    "difficulty": [
        "fácil", "fácil", "médio", "fácil", "médio",
        "médio", "fácil", "médio", "fácil", "fácil",
        "fácil", "médio", "difícil", "médio", "fácil",
        "fácil", "fácil", "difícil", "fácil", "fácil"
    ]
}

recipes_df = pd.DataFrame(recipes_data)
print(f"✅ Recipe database loaded: {len(recipes_df)} recipes")
print(f"   Categories: {recipes_df['category'].unique()}")
recipes_df.head()

## 3. NLP Preprocessing

We use **spaCy's Portuguese model** to process user input:
removing stopwords, punctuation, and normalizing text.
This is critical for accurate ingredient extraction from natural language.

In [ ]:
def preprocess_text(text: str) -> str:
    """
    Cleans and normalizes Portuguese text using spaCy.
    Removes stopwords, punctuation, and non-alphabetic tokens.
    Returns a clean string of lemmatized tokens.
    """
    doc = nlp(text.lower())
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and token.is_alpha
        and len(token.text) > 2
    ]
    return " ".join(tokens)


def extract_ingredients(user_text: str) -> list:
    """
    Extracts likely ingredient tokens from a free-text user message.
    Uses spaCy tokenization + stopword removal.
    """
    doc = nlp(user_text.lower())
    ingredients = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and token.is_alpha
        and len(token.text) > 2
    ]
    return ingredients


# Test the extraction
test_input = "Tenho frango, cenoura e um pouco de arroz sobrando na geladeira"
extracted = extract_ingredients(test_input)
print(f"Input: '{test_input}'")
print(f"Extracted tokens: {extracted}")

## 4. TF-IDF Recommendation Engine

Instead of simple keyword matching, ChefBot uses **TF-IDF vectorization**
combined with **cosine similarity** to rank recipes.

This approach handles partial matches and ingredient variations much better
than exact string matching — a user saying "galinha" instead of "frango"
will still get relevant results because of the vector space representation.

In [ ]:
# Build TF-IDF matrix from recipe ingredient lists
tfidf_vectorizer = TfidfVectorizer()
recipe_matrix = tfidf_vectorizer.fit_transform(recipes_df["ingredients"])

print(f"✅ TF-IDF matrix built: {recipe_matrix.shape[0]} recipes × {recipe_matrix.shape[1]} ingredient terms")


def recommend_recipes(user_input: str, df: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
    """
    Recommends top_n recipes based on cosine similarity between
    the user's available ingredients and the recipe ingredient vectors.

    Args:
        user_input: Free-text description of available ingredients
        df: Recipe DataFrame
        top_n: Number of recommendations to return

    Returns:
        DataFrame with recommended recipes sorted by similarity score
    """
    # Preprocess user input
    processed_input = preprocess_text(user_input)

    # Vectorize user input using the fitted TF-IDF vectorizer
    user_vector = tfidf_vectorizer.transform([processed_input])

    # Calculate cosine similarity between user vector and all recipes
    similarities = cosine_similarity(user_vector, recipe_matrix).flatten()

    # Add similarity scores to dataframe and sort
    df_scored = df.copy()
    df_scored["similarity_score"] = similarities
    df_scored["similarity_pct"] = (similarities * 100).round(1)

    recommendations = df_scored.sort_values("similarity_score", ascending=False).head(top_n)

    return recommendations[["name", "category", "prep_time_min", "difficulty", "ingredients", "similarity_pct"]]


# Test recommendation
user_input = "Tenho ovos, espinafre, queijo e um pouco de azeite"
results = recommend_recipes(user_input, recipes_df)
print(f"\n🔍 User input: '{user_input}'")
print(f"\n🍽️ Top 3 recommended recipes:")
print(results.to_string(index=False))

## 5. Full Chatbot Interaction Flow

This simulates the conversational flow that, in production, runs via
**WhatsApp Business API** with **LangChain** managing the dialogue state.

In [ ]:
def chatbot_response(user_message: str) -> str:
    """
    Simulates ChefBot's full response pipeline:
    1. Extract ingredients from user message
    2. Run TF-IDF recommendation
    3. Format a friendly response
    """
    ingredients = extract_ingredients(user_message)

    if not ingredients:
        return "Não consegui identificar ingredientes na sua mensagem. Tente algo como: 'Tenho ovos, tomate e queijo'."

    recommendations = recommend_recipes(user_message, recipes_df, top_n=3)
    top = recommendations.iloc[0]

    response = f"""
🤖 ChefBot detectou os ingredientes: {', '.join(ingredients)}

✅ Melhor recomendação para você:
   📌 {top['name']} ({top['category']})
   ⏱️ Tempo de preparo: {top['prep_time_min']} minutos
   📊 Nível: {top['difficulty']}
   🎯 Compatibilidade: {top['similarity_pct']}%

Outras opções:"""

    for _, row in recommendations.iloc[1:].iterrows():
        response += f"\n   • {row['name']} — {row['similarity_pct']}% compatível"

    response += "\n\n💡 Lembre-se: usar o que você já tem evita desperdício!"

    return response


# Simulate interactions
messages = [
    "Tenho arroz, frango e cenoura sobrando. O que posso fazer?",
    "Só tenho ovos e queijo aqui em casa",
    "Encontrei batata, alecrim e azeite na despensa"
]

for msg in messages:
    print("=" * 60)
    print(f"👤 Usuário: {msg}")
    print(chatbot_response(msg))
print("=" * 60)

## 6. KPI Simulation

In the production version, all interactions are logged to **MongoDB**
and visualized in a **Power BI dashboard**. Below we simulate 90 days
of interaction data to illustrate the KPI framework.

In [ ]:
# Simulate 90 days of ChefBot interactions
random.seed(42)
np.random.seed(42)

n_interactions = 450
base_date = datetime(2024, 10, 1)

simulated_logs = pd.DataFrame({
    "timestamp": [base_date + timedelta(days=random.randint(0, 89),
                                        hours=random.randint(6, 22))
                  for _ in range(n_interactions)],
    "user_id": [f"user_{random.randint(1, 80):03d}" for _ in range(n_interactions)],
    "recommended_recipe": random.choices(recipes_df["name"].tolist(), k=n_interactions),
    "similarity_score": np.random.beta(5, 2, n_interactions).round(3),
    "user_rated_helpful": np.random.choice([True, False], n_interactions, p=[0.78, 0.22]),
    "ingredients_count": np.random.randint(2, 7, n_interactions),
    "waste_avoided_g": np.random.randint(50, 400, n_interactions)
})

# Calculate KPIs
total_interactions = len(simulated_logs)
unique_users = simulated_logs["user_id"].nunique()
satisfaction_rate = simulated_logs["user_rated_helpful"].mean() * 100
avg_similarity = simulated_logs["similarity_score"].mean() * 100
total_waste_avoided_kg = simulated_logs["waste_avoided_g"].sum() / 1000
avg_waste_per_session = simulated_logs["waste_avoided_g"].mean()

print("=" * 50)
print("📊 CHEFBOT — KPI DASHBOARD (90-day simulation)")
print("=" * 50)
print(f"  Total interactions:       {total_interactions:,}")
print(f"  Unique users:             {unique_users}")
print(f"  Satisfaction rate:        {satisfaction_rate:.1f}%")
print(f"  Avg recommendation match: {avg_similarity:.1f}%")
print(f"  Total waste avoided:      {total_waste_avoided_kg:.1f} kg")
print(f"  Avg waste/session:        {avg_waste_per_session:.0f}g")
print("=" * 50)

# Most recommended recipes
print("\n🏆 Top 5 Most Recommended Recipes:")
top_recipes = simulated_logs["recommended_recipe"].value_counts().head(5)
for i, (recipe, count) in enumerate(top_recipes.items(), 1):
    print(f"  {i}. {recipe}: {count} times")

# Weekly trend
simulated_logs["week"] = simulated_logs["timestamp"].dt.isocalendar().week
weekly = simulated_logs.groupby("week").agg(
    interactions=("user_id", "count"),
    avg_satisfaction=("user_rated_helpful", "mean")
).round(3)
print("\n📅 Weekly Interaction Trend:")
print(weekly.to_string())

## 7. Architecture Overview

```
WhatsApp User
     │
     ▼
WhatsApp Business API
     │
     ▼
LangChain Conversation Manager
     │
     ├── spaCy NLP (ingredient extraction)
     │
     ├── TF-IDF Engine (recipe recommendation)
     │
     └── MongoDB (recipe storage + interaction logs)
               │
               ▼
         Power BI Dashboard (KPIs)
```

**Production stack:** Python · spaCy (pt_core_news_sm) · scikit-learn ·
LangChain · MongoDB · WhatsApp Business API · Power BI

---
*ChefBot — MBA Final Project | Ana Isabel Assunção*
*[LinkedIn](https://www.linkedin.com/in/ana-isabel-assuncao-ap/) |
[GitHub](https://github.com/anaisabelaap)*